# Phase-2 Step 4: Multimodal Model

**Objective:** Implement and train the full multimodal SOH prediction model.

**Architecture:**
1. **Numeric Encoder**: CNN-LSTM for temporal features
2. **Feature-to-Text**: Rule-based conversion of physics features
3. **LLM Embeddings**: Sentence-BERT (frozen)
4. **Semantic Encoder**: MLP for semantic features
5. **Cross-Attention Fusion**: Combine modalities
6. **SOH Regressor**: Final prediction

**Comparison:**
- Full Multimodal (Numeric + Semantic)
- Numeric Only (for baseline comparison)

In [ ]:
# --- Portable setup (Windows / macOS / Linux) — do not hardcode paths ---
import importlib.util
from pathlib import Path

def _load_notebook_setup():
    cwd = Path.cwd().resolve()
    candidates = [
        cwd / "notebooks" / "notebook_setup.py",
        cwd / "notebook_setup.py",
    ]
    if cwd.name == "notebooks":
        candidates.insert(0, cwd / "notebook_setup.py")
    for fp in candidates:
        if fp.is_file():
            spec = importlib.util.spec_from_file_location("_notebook_setup", fp)
            mod = importlib.util.module_from_spec(spec)
            assert spec.loader is not None
            spec.loader.exec_module(mod)
            return mod
    raise FileNotFoundError(
        "Missing notebooks/notebook_setup.py. Unzip the full project, open a terminal, "
        "cd into the repo root, then run: jupyter notebook"
    )

_ns = _load_notebook_setup()
PROJECT_ROOT = _ns.setup()
print(f"PROJECT_ROOT = {PROJECT_ROOT}")


import sys
import os

# ============================================================================
# SETUP
# ============================================================================


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
import torch

from multimodal_model import (
    MultimodalTrainer,
    MultimodalSOHModel,
    NumericOnlyModel,
    LLMEmbedder,
    set_seed
)
from feature_to_text import FeatureToText, convert_features_to_text
from utils import plot_style_setup

plot_style_setup()

# Load config
with open('phase2_config.yaml', 'r') as f:
    config = yaml.safe_load(f)

set_seed(config['random_seed'])

print("✓ Imports complete")
print(f"✓ PyTorch: {torch.__version__}")
print(f"✓ Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 1. Load Harmonized Data

In [ ]:
# Load train and test features
train_path = 'data/phase2/harmonized/train_features.csv'
test_path = 'data/phase2/harmonized/test_features.csv'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print(f"Train: {len(train_df)} samples")
print(f"Test: {len(test_df)} samples")

# Feature columns
exclude_cols = ['battery_id', 'cycle_number', 'original_index', 'soh', 'has_spikes']
feature_cols = [c for c in train_df.columns if c not in exclude_cols]

print(f"\nFeatures ({len(feature_cols)}):")
for col in feature_cols:
    print(f"  - {col}")

## 2. Feature-to-Text Conversion Demo

In [ ]:
# Test feature-to-text conversion
converter = FeatureToText()

# Convert a few samples
sample_indices = [0, len(train_df)//2, len(train_df)-1]  # early, mid, late

print("="*80)
print("FEATURE-TO-TEXT CONVERSION EXAMPLES")
print("="*80)

for idx in sample_indices:
    row = train_df.iloc[idx]
    desc = converter.convert_single(row.to_dict())
    
    print(f"\n--- Cycle {desc.cycle_number} (SOH: {desc.soh:.1f}%) ---")
    print(f"\nFull Description:")
    print(desc.full_description)
    print(f"\nDiagnosis: {desc.diagnosis}")
    print("-"*80)

In [ ]:
# Convert all training data to text
train_text_df = convert_features_to_text(train_df)

print(f"\nConverted {len(train_text_df)} cycles to text descriptions")
print(f"\nColumns: {train_text_df.columns.tolist()}")

# Show sample
train_text_df[['cycle_number', 'soh', 'soh_text', 'diagnosis']].head()

## 3. LLM Embedding Generation

In [ ]:
# Initialize LLM embedder
embedder = LLMEmbedder(model_name=config['multimodal']['llm_embeddings']['model'])

# Load the model
embedder.load_model()

In [ ]:
# Test embedding generation on a few samples
sample_texts = train_text_df['full_description'].iloc[:5].tolist()

print("Generating test embeddings...")
sample_embeddings = embedder.embed_texts(sample_texts)

print(f"\nEmbedding shape: {sample_embeddings.shape}")
print(f"Embedding dimension: {embedder.embedding_dim}")
print(f"\nSample embedding stats:")
print(f"  Mean: {sample_embeddings.mean():.4f}")
print(f"  Std: {sample_embeddings.std():.4f}")
print(f"  Min: {sample_embeddings.min():.4f}")
print(f"  Max: {sample_embeddings.max():.4f}")

## 4. Initialize Multimodal Trainer

In [ ]:
# Create trainer
trainer = MultimodalTrainer(config, PROJECT_ROOT)

print("\nMultimodal configuration:")
print(f"  Numeric encoder output: {config['multimodal']['numeric_encoder']['output_dim']}")
print(f"  LLM model: {config['multimodal']['llm_embeddings']['model']}")
print(f"  Fusion type: {config['multimodal']['fusion']['type']}")
print(f"  Fusion output: {config['multimodal']['fusion']['output_dim']}")
print(f"  Attention heads: {config['multimodal']['fusion']['attention_heads']}")

## 5. Train Multimodal Models

In [ ]:
# Run experiments
# This trains:
# 1. Full Multimodal (Numeric + Semantic) - 3 runs
# 2. Numeric Only (for comparison) - 3 runs

results_df = trainer.run_experiments(
    train_df, 
    test_df, 
    feature_cols,
    num_runs=config['training']['num_runs']
)

## 6. Analyze Results

In [ ]:
# Display all results
print("\nAll Experiment Results:")
results_df

In [ ]:
# Summary statistics
summary = results_df.groupby('model').agg({
    'mae': ['mean', 'std'],
    'rmse': ['mean', 'std'],
    'r2': ['mean', 'std']
}).round(4)

print("\nSummary (mean ± std):")
summary

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

metrics = ['mae', 'rmse', 'r2']
titles = ['MAE (lower is better)', 'RMSE (lower is better)', 'R² (higher is better)']

for ax, metric, title in zip(axes, metrics, titles):
    means = results_df.groupby('model')[metric].mean()
    stds = results_df.groupby('model')[metric].std()
    
    x = np.arange(len(means))
    colors = ['#e74c3c' if 'numeric' in m else '#2ecc71' for m in means.index]
    
    bars = ax.bar(x, means, yerr=stds, capsize=5, color=colors, edgecolor='black')
    
    ax.set_xticks(x)
    ax.set_xticklabels([m.replace('_', '\n') for m in means.index], fontsize=10)
    ax.set_ylabel(metric.upper())
    ax.set_title(title, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    
    # Add value labels
    for bar, mean, std in zip(bars, means, stds):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + std + 0.01,
                f'{mean:.4f}', ha='center', fontsize=9)

plt.suptitle('Multimodal vs Numeric-Only Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('results/phase2/figures/multimodal_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Saved: results/phase2/figures/multimodal_comparison.png")

In [ ]:
# Calculate improvement
mm_results = results_df[results_df['model'] == 'multimodal_full']
num_results = results_df[results_df['model'] == 'numeric_only']

print("\n" + "="*60)
print("MULTIMODAL IMPROVEMENT ANALYSIS")
print("="*60)

for metric in ['mae', 'rmse']:
    mm_val = mm_results[metric].mean()
    num_val = num_results[metric].mean()
    improvement = (num_val - mm_val) / num_val * 100
    
    print(f"\n{metric.upper()}:")
    print(f"  Numeric-only: {num_val:.4f}")
    print(f"  Multimodal:   {mm_val:.4f}")
    print(f"  Improvement:  {improvement:.2f}%")

# R² improvement
mm_r2 = mm_results['r2'].mean()
num_r2 = num_results['r2'].mean()
r2_improvement = mm_r2 - num_r2

print(f"\nR²:")
print(f"  Numeric-only: {num_r2:.4f}")
print(f"  Multimodal:   {mm_r2:.4f}")
print(f"  Improvement:  +{r2_improvement:.4f}")

print("\n" + "="*60)

## 7. Prediction Visualization

In [ ]:
# Get predictions from the best multimodal model
from torch.utils.data import DataLoader
from multimodal_model import MultimodalDataset

# Get test predictions
best_model = trainer.best_model
test_loader = trainer.data['test_loader']

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
best_model.eval()

all_preds = []
all_targets = []

with torch.no_grad():
    for numeric, semantic, target in test_loader:
        numeric = numeric.to(device)
        semantic = semantic.to(device)
        
        output = best_model(numeric, semantic)
        
        all_preds.extend(output.cpu().numpy())
        all_targets.extend(target.numpy())

preds = np.array(all_preds)
targets = np.array(all_targets)

print(f"Predictions: {len(preds)}")

In [ ]:
# Prediction vs Actual
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot
ax = axes[0]
ax.scatter(targets, preds, alpha=0.6, s=30)

min_val = min(targets.min(), preds.min())
max_val = max(targets.max(), preds.max())
ax.plot([min_val, max_val], [min_val, max_val], 'r--', label='Perfect', linewidth=2)

# Compute R²
ss_res = np.sum((targets - preds) ** 2)
ss_tot = np.sum((targets - np.mean(targets)) ** 2)
r2 = 1 - ss_res / ss_tot

ax.set_xlabel('Actual SOH (%)', fontsize=12)
ax.set_ylabel('Predicted SOH (%)', fontsize=12)
ax.set_title(f'Multimodal Model: Predicted vs Actual\nR² = {r2:.4f}', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Error distribution
ax = axes[1]
errors = preds - targets

ax.hist(errors, bins=30, edgecolor='black', alpha=0.7, color='#3498db')
ax.axvline(x=0, color='r', linestyle='--', linewidth=2, label='Zero error')
ax.axvline(x=np.mean(errors), color='g', linestyle='--', linewidth=2,
           label=f'Mean: {np.mean(errors):.3f}')

ax.set_xlabel('Prediction Error (%)', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title(f'Error Distribution\nStd: {np.std(errors):.3f}', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/phase2/figures/multimodal_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Saved: results/phase2/figures/multimodal_predictions.png")

## 8. Model Architecture Visualization

In [ ]:
# Print model architecture
print("\n" + "="*60)
print("MULTIMODAL MODEL ARCHITECTURE")
print("="*60)
print(best_model)
print("="*60)

# Count parameters
total_params = sum(p.numel() for p in best_model.parameters())
trainable_params = sum(p.numel() for p in best_model.parameters() if p.requires_grad)

print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## 9. Save Results

In [ ]:
# Save all results
trainer.save_results(results_df)

In [ ]:
# Print final summary
trainer.print_summary(results_df)

## 10. Comparison with Baselines

In [ ]:
# Load baseline results for comparison
import glob

baseline_files = glob.glob('models/phase2/baselines/baseline_results_*.csv')

if baseline_files:
    baseline_df = pd.read_csv(sorted(baseline_files)[-1])  # Latest
    
    # Combine all results
    all_results = pd.concat([baseline_df, results_df], ignore_index=True)
    
    # Summary
    print("\n" + "="*80)
    print("COMPLETE MODEL COMPARISON (Baselines + Multimodal)")
    print("="*80)
    
    summary = all_results.groupby('model').agg({
        'mae': ['mean', 'std'],
        'rmse': ['mean', 'std'],
        'r2': ['mean', 'std']
    }).round(4)
    
    print(f"\n{'Model':<20} {'MAE':<18} {'RMSE':<18} {'R²':<18}")
    print('─' * 75)
    
    for model in summary.index:
        mae = f"{summary.loc[model, ('mae', 'mean')]:.4f} ± {summary.loc[model, ('mae', 'std')]:.4f}"
        rmse = f"{summary.loc[model, ('rmse', 'mean')]:.4f} ± {summary.loc[model, ('rmse', 'std')]:.4f}"
        r2 = f"{summary.loc[model, ('r2', 'mean')]:.4f} ± {summary.loc[model, ('r2', 'std')]:.4f}"
        
        print(f"{model:<20} {mae:<18} {rmse:<18} {r2:<18}")
    
    print('─' * 75)
    
    # Find best model
    best_rmse = all_results.groupby('model')['rmse'].mean().idxmin()
    best_val = all_results.groupby('model')['rmse'].mean().min()
    
    print(f"\n✓ Best model: {best_rmse} (RMSE = {best_val:.4f})")
    print("="*80)
else:
    print("No baseline results found. Run Step 3 first.")

## Summary

**Step 4 Completed:**

✓ Feature-to-text conversion implemented (rule-based)  
✓ LLM embeddings generated (Sentence-BERT)  
✓ Multimodal model trained (CNN-LSTM + Cross-Attention)  
✓ Numeric-only baseline for comparison  
✓ Results compared with Step 3 baselines  

**Key Findings:**
- Multimodal model combines numeric and semantic features
- Cross-attention fusion learns modality interactions
- Compare improvement % over numeric-only

**Output Files:**
- `models/phase2/multimodal/multimodal_best_*.pt`
- `models/phase2/multimodal/multimodal_results_*.csv`
- `models/phase2/multimodal/text_descriptions_*.csv`
- `results/phase2/figures/multimodal_*.png`

**Next Step:** Step 5 - Ablation Studies